# 🎓 StudentGPT — Custom LoRA Fine-Tuning & GGUF Exporter (Free Google Colab T4 GPU)

This notebook fine-tunes **`Qwen2.5-3B-Instruct`** (or `7B`) using **4-bit QLoRA (`Unsloth` + `TRL SFTTrainer`)** on your **245-example `studentgpt_sft_dataset.jsonl`** dataset and exports it to **Hugging Face Hub (LoRA + 4-bit GGUF)** for 24/7 hosting.

### How to Run (100% Free):
1. In Google Colab, click **Runtime → Change runtime type → T4 GPU**.
2. Upload `studentgpt_sft_dataset.jsonl` (from your `model-training/` folder) to the Colab Files sidebar on the left.
3. Click **Runtime → Run all**!

In [ ]:
# Step 1: Install Unsloth, Hugging Face TRL, PEFT, and BitsAndBytes
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes xformers datasets matplotlib

In [ ]:
# Step 2: Load 4-Bit Quantized Qwen2.5-7B-Instruct (16K Context) & Inject LoRA Adapters
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported

max_seq_length = 16384
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
model.print_trainable_parameters()

In [ ]:
# Step 3: Load StudentGPT 245-Example SFT Dataset & Train with SFTTrainer
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

dataset = load_dataset("json", data_files="studentgpt_sft_dataset.jsonl", split="train")

def format_chat(batch):
    return {
        "text": [
            tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
            for msgs in batch["messages"]
        ]
    }

dataset = dataset.map(format_chat, batched=True)
print(f"Loaded {len(dataset)} StudentGPT training examples!")

import inspect
sft_tok_arg = {"processing_class": tokenizer} if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters else {"tokenizer": tokenizer}
trainer = SFTTrainer(
    model = model,
    **sft_tok_arg,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 35,
        learning_rate = 8e-5,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "studentgpt-lora-outputs",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# Step 4: Plot Training Loss Curve (Save this chart for your FYP Report / Presentation!)
import matplotlib.pyplot as plt

steps = [x["step"] for x in trainer.state.log_history if "loss" in x]
losses = [x["loss"] for x in trainer.state.log_history if "loss" in x]

plt.figure(figsize=(8, 4.5))
plt.plot(steps, losses, marker="o", color="#724060", linewidth=2.5)
plt.title("StudentGPT QLoRA Fine-Tuning Loss Curve", fontsize=14, fontweight="bold")
plt.xlabel("Training Step")
plt.ylabel("Cross-Entropy Loss")
plt.grid(True, alpha=0.3)
plt.savefig("studentgpt_training_loss.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Step 5: Test Your Newly Fine-Tuned StudentGPT Model!
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are StudentGPT, a custom fine-tuned academic AI engine created by Hafsa Tahir."},
    {"role": "user", "content": "Who are you and who created you?"}
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.3)
print(tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True))

In [ ]:
# Step 6: Save LoRA Weights & Export 4-Bit GGUF to Hugging Face Hub for 24/7 Space Hosting
model.save_pretrained("studentgpt_lora_model")
tokenizer.save_pretrained("studentgpt_lora_model")

# Paste your free Hugging Face Write Token (from https://huggingface.co/settings/tokens) to publish:
HF_TOKEN = "" # e.g. "hf_xxxxxxxxxxxxxxxxx"
HF_USERNAME = "hafsa-tahir"

if HF_TOKEN:
    # 1. Push LoRA Adapter weights
    model.push_to_hub(f"{HF_USERNAME}/StudentGPT-Academic-LoRA", token=HF_TOKEN)
    tokenizer.push_to_hub(f"{HF_USERNAME}/StudentGPT-Academic-LoRA", token=HF_TOKEN)
    # 2. Export & Push 4-bit GGUF model for 24/7 Hugging Face Space / Ollama
    model.push_to_hub_gguf(
        f"{HF_USERNAME}/StudentGPT-Academic-GGUF",
        tokenizer,
        quantization_method="q4_k_m",
        token=HF_TOKEN,
    )
    print("Published both LoRA Adapter and GGUF model to Hugging Face Hub!")